# Circle Manifold Density

Chapter 17: Score Function and Energy-Based Models · *Mathematics for AI and Machine Learning* by Fuheng Wu

Density concentrated on the unit circle; on vs off manifold.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-17/manifold-density/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

# The demo's density concentrated near the unit circle, width sigma
sigma = 0.05


def unnormalized(x):
    r2 = (x ** 2).sum(-1)
    return torch.exp(-((r2 - 1) ** 2) / (2 * sigma ** 2))


# ── On vs off the manifold ──────────────────────────────────────────────────
pts = torch.tensor([[1.0, 0.0], [0.0, -1.0], [0.5, 0.0], [1.1, 0.0], [0.0, 0.0]], dtype=torch.float64)
for p_, v in zip(pts.tolist(), unnormalized(pts).tolist()):
    print(f"x = {p_}: |x| = {math.hypot(*p_):.2f}, density (unnormalized) = {v:.3e}")

# ── Normalize on a grid and measure where the mass is ───────────────────────
g = torch.linspace(-1.6, 1.6, 1601, dtype=torch.float64)
X, Y = torch.meshgrid(g, g, indexing="xy")
grid = torch.stack([X, Y], dim=-1)
dens = unnormalized(grid)
cell = (g[1] - g[0]) ** 2
Z = dens.sum() * cell
r = grid.norm(dim=-1)
band = (r - 1).abs() < 3 * sigma / 2          # |r^2 - 1| < 3 sigma, i.e. |r - 1| ~ 1.5 sigma
print(f"\nZ = {Z.item():.5f};  fraction of mass within the thin band around the circle: "
      f"{(dens[band].sum() * cell / Z).item():.4f}")
print(f"that band covers only {(band.double().sum() * cell).item() / (3.2 ** 2):.3%} of the plotted square")

# ── The score points back to the circle, and gets huge as sigma shrinks ─────
x = torch.tensor([1.2, 0.0], dtype=torch.float64, requires_grad=True)
(s,) = torch.autograd.grad(torch.log(unnormalized(x)), x)
print(f"\nscore at (1.2, 0) = {s.tolist()} (points inward); formula -2 x (|x|^2 - 1) / sigma^2 = "
      f"{(-2 * x.detach() * (x.detach() @ x.detach() - 1) / sigma ** 2).tolist()}")

x = [1.0, 0.0]: |x| = 1.00, density (unnormalized) = 1.000e+00
x = [0.0, -1.0]: |x| = 1.00, density (unnormalized) = 1.000e+00
x = [0.5, 0.0]: |x| = 0.50, density (unnormalized) = 1.386e-49
x = [1.1, 0.0]: |x| = 1.10, density (unnormalized) = 1.477e-04
x = [0.0, 0.0]: |x| = 0.00, density (unnormalized) = 1.384e-87

Z = 0.39374;  fraction of mass within the thin band around the circle: 0.9971
that band covers only 9.203% of the plotted square

score at (1.2, 0) = [-422.39999999999986, -0.0] (points inward); formula -2 x (|x|^2 - 1) / sigma^2 = [-422.39999999999986, -0.0]
